In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Path to dataset files: /kaggle/input/steam-games-dataset


In [2]:
import pandas as pd
import pathlib as path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
downloaded_dataset_dir = "/content/datasets/fronkongames/steam-games-dataset/versions/31"
import os
print(os.listdir(downloaded_dataset_dir))

df = pd.read_csv(downloaded_dataset_dir + "/games.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets/fronkongames/steam-games-dataset/versions/31'

In [ ]:
df.info()

In [ ]:
features_to_keep_initial = ['Price', 'Windows', 'Mac', 'Linux', 'Positive', 'Negative', 'Achievements']
df_clean = df[features_to_keep_initial].copy()

df_clean['Is_Hit'] = (df_clean['Positive'] > 500).astype(int)

df_clean['Windows'] = df_clean['Windows'].astype(int)
df_clean['Mac'] = df_clean['Mac'].astype(int)
df_clean['Linux'] = df_clean['Linux'].astype(int)

df_clean = df_clean.dropna()

print(df_clean.info())

In [ ]:
X = df_clean.drop(['Is_Hit', 'Positive', 'Negative'], axis=1)
y = df_clean['Is_Hit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(df_clean['Is_Hit'].value_counts())
model1 = DecisionTreeClassifier(random_state=42)
model2 = KNeighborsClassifier(n_neighbors=5)
model3 = LogisticRegression(max_iter=1000, random_state=42)

ensemble_model = VotingClassifier(
    estimators=[('dt', model1), ('knn', model2), ('lr', model3)],
    voting='hard'
)

ensemble_model.fit(X_train, y_train)

y_pred = ensemble_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
joblib.dump(ensemble_model, 'steam_hit_predictor.pkl')

In [ ]:
import joblib
import pandas as pd


file_path = 'steam_hit_predictor.pkl'

try:
    # ใช้ joblib โหลดแทน pickle
    model = joblib.load(file_path)
    print("โหลดโมเดลสำเร็จ!")
    print(type(model))
except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")

# ดูรายชื่อโมเดลย่อยทั้งหมด
print(model.estimators_)

# หรือดูชื่อที่ตั้งไว้พร้อมกับตัวโมเดล
for name, est in model.named_estimators_.items():
    print(f"Model Name: {name}")
    print(f"Details: {est}")
    print("-" * 20)

In [ ]:
import pandas as pd
import joblib

# สมมติว่าโหลดโมเดลมาไว้ในตัวแปร model แล้ว
# model = joblib.load('steam_hit_predictor.pkl')

# 1. จำลองข้อมูลโดย "ตัด" Positive และ Negative ออกไป
new_game = pd.DataFrame({
    'Price': [299],          # ราคา
    'Windows': [1],          # รองรับ Windows (1 = รองรับ, 0 = ไม่)
    'Mac': [0],              # รองรับ Mac
    'Linux': [0],            # รองรับ Linux
    'Achievements': [35]     # จำนวน Achievements
})

print("ข้อมูลเกมที่ต้องการทำนาย:")
print(new_game.to_string(index=False))
print("-" * 30)

try:
    # 2. ให้โมเดลทำนาย
    result = model.predict(new_game)

    # 3. แปลผลลัพธ์
    if result[0] == 1:
        print("🎯 ผลการทำนาย: เกมนี้มีโอกาสเป็น 'Hit' (เกมฮิต) สูงมากครับ!")
    else:
        print("📉 ผลการทำนาย: เกมนี้อาจจะยังไม่ถึงระดับ 'Hit' ครับ")

except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")

    # ทริคเพิ่มเติม: ถ้ายังพังอีก ให้โมเดลบอกมาเลยว่ามันอยากได้คอลัมน์อะไรบ้าง
    if hasattr(model, 'feature_names_in_'):
        print(f"คอลัมน์ที่โมเดลต้องการจริงๆ คือ: {model.feature_names_in_}")